In [12]:
import sys
from pathlib import Path

# Asumsikan kamu menjalankan notebook dari 'data_pipeline_pyspark/notebooks'
# dan kamu ingin import dari 'data_pipeline_pyspark/src'
project_root = Path.cwd().parent
sys.path.append(str(project_root))
import pandas as pd

from src.utils.helper import startup_investments_engine_pyspark

from src.staging.extract.extract_db import extract_database
from src.staging.extract.extract_db_pyspark import extract_database as extract_database_pyspark
from src.staging.extract.extract_spreadsheet_pyspark import extract_sheet_spark,extract_spreadsheet as extract_spreadsheet_pyspark
from src.staging.extract.extract_api_pyspark import extract_api_milestones_spark,extract_api_spark,extract_backfilling_spark,extract_api_milestones_spark

from src.staging.extract.extract_spreadsheet import extract_spreadsheet
from src.staging.extract.extract_api import extract_api_milestones,extract_backfilling
from src.staging.load.load import load_staging
from src.staging.extract.extract_spreadsheet import extract_spreadsheet, extract_sheet

from src.warehouse.extract.extract_db import extract_database as extract_staging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from src.staging.load.load_pyspark import load_staging_pyspark_upsert
from datetime import datetime
from src.warehouse.extract.extract_db_pyspark import extract_database as extract_db_pyspark_staging

from src.warehouse.extract.extract_db_pyspark import extract_database as extract_db_pyspark_staging

from src.warehouse.transform.dim_company_pyspark import transform_dim_company_spark
from src.warehouse.transform.dim_people_pyspark import transform_dim_people_spark
from src.warehouse.transform.dim_relationship_pyspark import transform_dim_relationship_spark
from src.warehouse.transform.fact_acquisitions_pyspark import transform_fact_acquisitions_spark
from src.warehouse.transform.fact_acquisitions_pyspark import transform_fact_acquisitions_spark
from src.warehouse.transform.fact_funding_rounds_pyspark import transform_fact_funding_rounds_spark
from src.warehouse.transform.fact_funds_pyspark import transform_fact_funds_spark
from src.warehouse.transform.fact_ipos_pyspark import transform_fact_ipos_spark
from src.warehouse.transform.fact_milestones_pyspark import transform_fact_milestones_spark
from src.warehouse.transform.fact_investments_pyspark import transform_fact_investments_spark
from src.warehouse.load.load_pyspark import load_warehouse_pyspark_upsert
from src.warehouse.validation.validation import report_validation


# Declare spark

In [2]:
from pyspark.sql import SparkSession

# Membuat SparkSession dengan semua core lokal dan paksa port 4041
spark = SparkSession.builder \
    .appName("Spark_Pipeline_Startup_Investments") \
    .config("spark.ui.showConsoleProgress", "false") \
    .master("spark://spark-master:7077") \
    .getOrCreate()
print("SparkSession is running...")
print(f"Spark Version: {spark.version}")



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SparkSession is running...
Spark Version: 3.4.3


# Staging

## Extract

### api

In [3]:
df_staging_api = extract_api_milestones_spark(spark, table_name='milestones')

+-------+----------+-------+------+----------+-------------------+---------+
|   step|   process| status|source|table_name|           etl_date|error_msg|
+-------+----------+-------+------+----------+-------------------+---------+
|staging|extraction|success|   api|milestones|2025-06-08 05:00:42|     null|
+-------+----------+-------+------+----------+-------------------+---------+



### spreadsheet

In [3]:
%%time
# relationship 
people_df = extract_spreadsheet_pyspark(spark,table_name='people')

# people 
relationships_df = extract_spreadsheet_pyspark(spark,table_name='relationships')

+-------+----------+-------+-----------+----------+-------------------+---------+
|   step|   process| status|     source|table_name|           etl_date|error_msg|
+-------+----------+-------+-----------+----------+-------------------+---------+
|staging|extraction|success|spreadsheet|    people|2025-06-08 05:09:56|     null|
+-------+----------+-------+-----------+----------+-------------------+---------+

+-------+----------+-------+-----------+-------------+-------------------+---------+
|   step|   process| status|     source|   table_name|           etl_date|error_msg|
+-------+----------+-------+-----------+-------------+-------------------+---------+
|staging|extraction|success|spreadsheet|relationships|2025-06-08 05:10:34|     null|
+-------+----------+-------+-----------+-------------+-------------------+---------+

CPU times: user 3.57 s, sys: 645 ms, total: 4.22 s
Wall time: 56.5 s


### DB

In [13]:
%%time
# acquisition
acquisition = extract_database_pyspark(spark,'acquisition')

#company
company = extract_database_pyspark(spark,'company')

#funding_rounds
funding_rounds = extract_database_pyspark(spark,'funding_rounds')

#funds
funds = extract_database_pyspark(spark,'funds')

#investments
investments = extract_database_pyspark(spark,'investments')

#ipos
ipos = extract_database_pyspark(spark,'ipos')


+-------+----------+-------+--------+-----------+-------------------+---------+
|   step|   process| status|  source| table_name|           etl_date|error_msg|
+-------+----------+-------+--------+-----------+-------------------+---------+
|staging|extraction|success|database|acquisition|2025-06-08 14:48:39|     null|
+-------+----------+-------+--------+-----------+-------------------+---------+

+-------+----------+-------+--------+----------+-------------------+---------+
|   step|   process| status|  source|table_name|           etl_date|error_msg|
+-------+----------+-------+--------+----------+-------------------+---------+
|staging|extraction|success|database|   company|2025-06-08 14:48:41|     null|
+-------+----------+-------+--------+----------+-------------------+---------+

+-------+----------+-------+--------+--------------+-------------------+---------+
|   step|   process| status|  source|    table_name|           etl_date|error_msg|
+-------+----------+-------+--------+

## Load

### spreadsheet

In [4]:
load_staging_pyspark_upsert(spark, data=people_df, schema='public', table_name='people', idx_name='people_id', source='spreadsheet')
load_staging_pyspark_upsert(spark, data=relationships_df, schema='public', table_name='relationships', idx_name='relationship_id', source='spreadsheet')


+-------+-------+-------+-----------+----------+--------------------+---------+
|   step|process| status|     source|table_name|            etl_date|error_msg|
+-------+-------+-------+-----------+----------+--------------------+---------+
|staging|   load|success|spreadsheet|    people|2025-06-08 05:10:...|     null|
+-------+-------+-------+-----------+----------+--------------------+---------+

+-------+-------+-------+-----------+-------------+--------------------+---------+
|   step|process| status|     source|   table_name|            etl_date|error_msg|
+-------+-------+-------+-----------+-------------+--------------------+---------+
|staging|   load|success|spreadsheet|relationships|2025-06-08 05:11:...|     null|
+-------+-------+-------+-----------+-------------+--------------------+---------+



### api

In [7]:
load_staging_pyspark_upsert(spark, data=df_staging_api, schema='public', table_name='milestones', idx_name='milestone_id', source='api')


+-------+-------+-------+------+----------+--------------------+---------+
|   step|process| status|source|table_name|            etl_date|error_msg|
+-------+-------+-------+------+----------+--------------------+---------+
|staging|   load|success|   api|milestones|2025-06-08 05:03:...|     null|
+-------+-------+-------+------+----------+--------------------+---------+



### DB

In [14]:
# acquisition
load_staging_pyspark_upsert(spark, data=acquisition, schema='public', table_name='acquisition', idx_name='acquisition_id', source='database')
#company
load_staging_pyspark_upsert(spark, data=company, schema='public', table_name='company', idx_name='object_id', source='database')

#funding_rounds
load_staging_pyspark_upsert(spark, data=funding_rounds, schema='public', table_name='funding_rounds', idx_name='funding_round_id', source='database')

#funds
load_staging_pyspark_upsert(spark, data=funds, schema='public', table_name='funds', idx_name='fund_id', source='database')

#investments
load_staging_pyspark_upsert(spark, data=investments, schema='public', table_name='investments', idx_name='investment_id', source='database')

#ipos
load_staging_pyspark_upsert(spark, data=ipos, schema='public', table_name='ipos', idx_name='ipo_id', source='database')


+-------+-------+-------+--------+-----------+--------------------+---------+
|   step|process| status|  source| table_name|            etl_date|error_msg|
+-------+-------+-------+--------+-----------+--------------------+---------+
|staging|   load|success|database|acquisition|2025-06-08 14:48:...|     null|
+-------+-------+-------+--------+-----------+--------------------+---------+

+-------+-------+-------+--------+----------+--------------------+---------+
|   step|process| status|  source|table_name|            etl_date|error_msg|
+-------+-------+-------+--------+----------+--------------------+---------+
|staging|   load|success|database|   company|2025-06-08 14:48:...|     null|
+-------+-------+-------+--------+----------+--------------------+---------+

+-------+-------+-------+--------+--------------+--------------------+---------+
|   step|process| status|  source|    table_name|            etl_date|error_msg|
+-------+-------+-------+--------+--------------+------------

# Warehouse

## people

In [5]:
%%time
## people
 
# extract
people_staging = extract_db_pyspark_staging(spark,table_name='people')

# transform
dim_people = transform_dim_people_spark(spark, people_staging,'people')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_people, table_name='dim_people', schema='public', 
               idx_name='people_nk', source='staging',table_process='people')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|    people|2025-06-08 05:11:20|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|    people|2025-06-08 05:11:22|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## company


In [10]:
## company
# extract
company_staging = extract_db_pyspark_staging(spark,'company')
# transform
dim_company = transform_dim_company_spark(spark,company_staging,'company')
# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_company, table_name='dim_company', schema='public', 
               idx_name='company_nk', source='staging',table_process='company')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|   company|2025-06-08 05:04:40|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|   company|2025-06-08 05:04:41|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## relationships


In [6]:
# relationships 
relationships_staging = extract_db_pyspark_staging(spark,table_name='relationships')

# transform
dim_relationships = transform_dim_relationship_spark(spark,relationships_staging,'relationships')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_relationships, table_name='dim_relationships', schema='public', 
               idx_name='relationship_nk', source='staging',table_process='relationships')

+---------+----------+-------+--------+-------------+-------------------+---------+
|     step|   process| status|  source|   table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-------------+-------------------+---------+
|warehouse|extraction|success|database|relationships|2025-06-08 05:11:43|     null|
+---------+----------+-------+--------+-------------+-------------------+---------+

+---------+--------------+-------+-------+-------------+-------------------+---------+
|     step|       process| status| source|   table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+-------------+-------------------+---------+
|warehouse|transformation|success|staging|relationships|2025-06-08 05:11:44|     null|
+---------+--------------+-------+-------+-------------+-------------------+---------+

+---------+-------+-------+-------+-------------+--------------------+---------+
|     step|process| status| source|   table_name|            e

## funding_rounds

In [12]:
# funding_rounds 
funding_rounds_staging = extract_db_pyspark_staging(spark,table_name='funding_rounds')

# transform
fact_funding_rounds = transform_fact_funding_rounds_spark(spark,funding_rounds_staging,'funding_rounds')

# load
load_warehouse_pyspark_upsert(spark=spark,data=fact_funding_rounds, table_name='fact_funding_rounds', schema='public', 
               idx_name='funding_round_nk', source='staging',table_process='funding_rounds')

+---------+----------+-------+--------+--------------+-------------------+---------+
|     step|   process| status|  source|    table_name|           etl_date|error_msg|
+---------+----------+-------+--------+--------------+-------------------+---------+
|warehouse|extraction|success|database|funding_rounds|2025-06-08 05:04:51|     null|
+---------+----------+-------+--------+--------------+-------------------+---------+

+---------+--------------+-------+-------+--------------+-------------------+---------+
|     step|       process| status| source|    table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+--------------+-------------------+---------+
|warehouse|transformation|success|staging|funding_rounds|2025-06-08 05:04:53|     null|
+---------+--------------+-------+-------+--------------+-------------------+---------+

+---------+-------+-------+-------+--------------+--------------------+---------+
|     step|process| status| source|    table_name| 

## funds


In [13]:
# funds 
funds_staging = extract_db_pyspark_staging(spark,table_name='funds')

# transform
fact_funds = transform_fact_funds_spark(spark,funds_staging,'funds')

# load
load_warehouse_pyspark_upsert(spark=spark,data=fact_funds, table_name='fact_funds', schema='public', 
               idx_name='fund_nk', source='staging',table_process='funds')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|     funds|2025-06-08 05:04:58|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|     funds|2025-06-08 05:04:59|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## ipos


In [14]:
# ipos 
ipos_staging = extract_db_pyspark_staging(spark,table_name='ipos')

# transform
fact_ipos = transform_fact_ipos_spark(spark,ipos_staging,'ipos')

# load
load_warehouse_pyspark_upsert(spark=spark,data=fact_ipos, table_name='fact_ipos', schema='public', 
               idx_name='ipo_nk', source='staging',table_process='ipos')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|      ipos|2025-06-08 05:05:05|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|      ipos|2025-06-08 05:05:06|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## acquisitions

In [15]:
# ipos 
acquisitions_staging = extract_db_pyspark_staging(spark,table_name='acquisition')

# transform
fact_acquisition = transform_fact_acquisitions_spark(spark,acquisitions_staging,'acquisition')

# load
load_warehouse_pyspark_upsert(spark=spark,data=fact_acquisition, table_name='fact_acquisition', schema='public', 
               idx_name='acquisition_nk', source='staging',table_process='acquisition')

+---------+----------+-------+--------+-----------+-------------------+---------+
|     step|   process| status|  source| table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-----------+-------------------+---------+
|warehouse|extraction|success|database|acquisition|2025-06-08 05:05:10|     null|
+---------+----------+-------+--------+-----------+-------------------+---------+

+---------+--------------+-------+-------+-----------+-------------------+---------+
|     step|       process| status| source| table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+-----------+-------------------+---------+
|warehouse|transformation|success|staging|acquisition|2025-06-08 05:05:12|     null|
+---------+--------------+-------+-------+-----------+-------------------+---------+

+---------+-------+-------+-------+-----------+--------------------+---------+
|     step|process| status| source| table_name|            etl_date|error_msg|
+----

## investments

In [ ]:
# ipos 
investments_staging = extract_db_pyspark_staging(spark,'investments')

# transform
fact_investments = transform_fact_investments_spark(spark,investments_staging,'investments')

# load
load_warehouse_pyspark_upsert(spark=spark,data=fact_investments, table_name='fact_investments', schema='public', 
               idx_name='investment_nk', source='staging',table_process='investments')

+---------+----------+-------+--------+-----------+-------------------+---------+
|     step|   process| status|  source| table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-----------+-------------------+---------+
|warehouse|extraction|success|database|investments|2025-06-08 05:05:18|     null|
+---------+----------+-------+--------+-----------+-------------------+---------+

+---------+--------------+-------+-------+-----------+-------------------+---------+
|     step|       process| status| source| table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+-----------+-------------------+---------+
|warehouse|transformation|success|staging|investments|2025-06-08 05:05:19|     null|
+---------+--------------+-------+-------+-----------+-------------------+---------+

+---------+-------+-------+-------+-----------+--------------------+---------+
|     step|process| status| source| table_name|            etl_date|error_msg|
+----

## milestones

In [17]:
# ipos 
milestones_staging = extract_db_pyspark_staging(spark,'milestones')

# transform
fact_milestones = transform_fact_milestones_spark(spark,milestones_staging,'milestones')

# load
load_warehouse_pyspark_upsert(spark=spark,data=fact_milestones, table_name='fact_milestones', schema='public', 
               idx_name='milestone_nk', source='staging',table_process='milestones')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|milestones|2025-06-08 05:05:26|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|milestones|2025-06-08 05:05:28|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

# Validation

In [18]:
from src.utils.helper import extract_target_pyspark

fact_milestones = extract_target_pyspark(spark,'fact_milestones')
fact_funding_rounds = extract_target_pyspark(spark,'fact_funding_rounds')
fact_investments = extract_target_pyspark(spark,'fact_investments')
fact_ipos = extract_target_pyspark(spark,'fact_ipos')
fact_funds = extract_target_pyspark(spark,'fact_funds')
fact_acquisition = extract_target_pyspark(spark,'fact_acquisition')
fact_investments = extract_target_pyspark(spark,'fact_investments')
dim_people = extract_target_pyspark(spark,'dim_people')
dim_relationships = extract_target_pyspark(spark,'dim_relationships')
dim_company = extract_target_pyspark(spark,'dim_company')


In [19]:
# validation
from src.warehouse.validation.validation import report_validation


## people
report_validation(table_name='people', df_spark=dim_people, id_col='people_nk')

## company
report_validation(table_name='company', df_spark=dim_company, id_col='company_nk')

## relationships
report_validation(table_name='relationships', df_spark=dim_relationships, id_col='relationship_nk')

## funding_rounds
report_validation(table_name='funding_rounds', df_spark=fact_funding_rounds, id_col='funding_round_nk')

## funds
report_validation(table_name='funds', df_spark=fact_funds, id_col='fund_nk')

## ipos
report_validation(table_name='ipos', df_spark=fact_ipos, id_col='ipo_nk')

## acquisitions
report_validation(table_name='acquisition', df_spark=fact_acquisition, id_col='acquisition_nk')

## investments
report_validation(table_name='investments', df_spark=fact_investments, id_col='investment_nk')

## milestones
report_validation(table_name='milestones', df_spark=fact_milestones, id_col='milestone_nk')


S3Error: S3 operation failed; code: InvalidAccessKeyId, message: The Access Key Id you provided does not exist in our records., resource: /validation-spark, request_id: 1846F8DD6FDA5372, host_id: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8, bucket_name: validation-spark